# Kaggle Titanic: Weighted Ensemble (Logistic + RandomForest + HistGB)

In [1]:
# 1) Imports and Paths
import os
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn import metrics
import joblib

# Paths and constants
DATA_DIR = "/home/atul-kumar/workspace/kaggle/titanic/data"
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")
SUBMISSION_PATH = os.path.join(DATA_DIR, "submission-ensemble.csv")
MODEL_DIR = os.path.join(DATA_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

RANDOM_STATE = 63
np.random.seed(RANDOM_STATE)
print("Paths set:", TRAIN_PATH, TEST_PATH)

Paths set: /home/atul-kumar/workspace/kaggle/titanic/data/train.csv /home/atul-kumar/workspace/kaggle/titanic/data/test.csv


In [2]:
# 2) Load Data
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
print("Train shape:", train_df.shape, " Test shape:", test_df.shape)
train_df.head(3)

Train shape: (891, 12)  Test shape: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [3]:
# 3) Enhanced Feature Engineering (same as RF/HGB)
def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Title + bucket
    out["Title"] = out["Name"].str.extract(r",\s*([^\.]+)\.")
    title_map = {
        'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
        'Lady': 'Rare', 'Countess': 'Rare', 'Dona': 'Rare', 'Sir': 'Rare', 'Don': 'Rare',
        'Jonkheer': 'Rare', 'Capt': 'Rare', 'Col': 'Rare', 'Dr': 'Rare', 'Rev': 'Rare',
        'Major': 'Rare'
    }
    out["TitleBucket"] = out["Title"].replace(title_map)
    out.loc[~out["TitleBucket"].isin(['Mr', 'Mrs', 'Miss', 'Master', 'Rare']), "TitleBucket"] = 'Rare'

    # Family
    out["FamilySize"] = out.get("SibSp", 0) + out.get("Parch", 0) + 1
    out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
    def _family_bin(n):
        if n == 1: return 'Single'
        if 2 <= n <= 4: return 'Small'
        return 'Large'
    out["FamilySizeBin"] = out["FamilySize"].apply(_family_bin)

    # Ticket
    if "Ticket" in out.columns:
        counts = out["Ticket"].value_counts()
        out["TicketGroup"] = out["Ticket"].map(counts)
        prefix = out["Ticket"].astype(str).str.replace(r"[^A-Za-z]+", "", regex=True).str.upper()
        prefix = prefix.replace("", np.nan).fillna("NONE")
        pref_counts = prefix.value_counts()
        common = set(pref_counts[pref_counts >= 10].index)
        prefix = prefix.where(prefix.isin(common), other="RARE")
        out["TicketPrefix"] = prefix
    else:
        out["TicketGroup"] = 1
        out["TicketPrefix"] = "NONE"

    # Cabin
    cabin = out.get("Cabin")
    out["CabinKnown"] = cabin.notna().astype(int)
    out["CabinDeck"] = cabin.astype(str).str[0]
    out["CabinDeck"] = out["CabinDeck"].where(out["CabinKnown"] == 1, other='U')

    # Fare/age transforms
    out["FareLog"] = np.log1p(out["Fare"]) if "Fare" in out.columns else 0.0
    out["AgePclass"] = out.get("Age", np.nan) * out.get("Pclass", np.nan)
    age_bins = [-1, 12, 18, 35, 60, 100]
    age_labels = ["child", "teen", "young", "adult", "senior"]
    out["AgeBand"] = pd.cut(out["Age"], bins=age_bins, labels=age_labels)

    return out

train_df_fe = add_engineered_features(train_df)
test_df_fe = add_engineered_features(test_df)
print("Engineered columns sample:", [c for c in ['TitleBucket','FamilySize','IsAlone','FamilySizeBin',
                                                 'TicketGroup','TicketPrefix','CabinKnown','CabinDeck',
                                                 'FareLog','AgePclass','AgeBand'] if c in train_df_fe.columns])

Engineered columns sample: ['TitleBucket', 'FamilySize', 'IsAlone', 'FamilySizeBin', 'TicketGroup', 'TicketPrefix', 'CabinKnown', 'CabinDeck', 'FareLog', 'AgePclass', 'AgeBand']


In [4]:
# 4) Preprocessing
base_features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
engineered = [
    "TitleBucket", "FamilySize", "IsAlone", "FamilySizeBin", "TicketGroup", "TicketPrefix",
    "CabinKnown", "CabinDeck", "FareLog", "AgePclass", "AgeBand"
]
all_features = base_features + engineered

numeric_features = ["Age", "SibSp", "Parch", "Fare", "FamilySize", 
                    "TicketGroup", "FareLog", "AgePclass"]
categorical_features = ["Pclass", "Sex", "Embarked", "TitleBucket", "FamilySizeBin", 
                        "TicketPrefix", "CabinKnown", "CabinDeck", "AgeBand"]

numeric_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
categorical_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")), 
                                          ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

preprocess = ColumnTransformer(transformers=[("num", numeric_transformer, numeric_features),
                                             ("cat", categorical_transformer, categorical_features)])

X = train_df_fe[all_features]
y = train_df_fe["Survived"]
X_test = test_df_fe[all_features]
print("Feature matrix shape:", X.shape)

Feature matrix shape: (891, 18)


In [5]:
# 5) Define three base models
logit = Pipeline(steps=[("pre", preprocess), ("clf", LogisticRegression(max_iter=1000, n_jobs=None))])
rf = Pipeline(steps=[
    ("pre", preprocess),
    ("clf", RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=2, min_samples_split=2, 
                                   max_features=0.6, bootstrap=True, random_state=RANDOM_STATE))])
hgb = Pipeline(steps=[
    ("pre", preprocess),
    ("clf", HistGradientBoostingClassifier(learning_rate=0.08, max_depth=4, max_leaf_nodes=31, 
                                           min_samples_leaf=20, l2_regularization=0.1, max_bins=255, random_state=RANDOM_STATE))])
models = {"logit": logit, "rf": rf, "hgb": hgb}

In [6]:
# 6) OOF predictions for each model
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
p_oof = {}
for name, mdl in models.items():
    p = cross_val_predict(mdl, X, y, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
    p_oof[name] = p
    auc = metrics.roc_auc_score(y, p)
    acc = metrics.accuracy_score(y, (p >= 0.5).astype(int))
    print(f"{name}: OOF AUC={auc:.4f} ACC@0.5={acc:.4f}")

/home/atul-kumar/miniconda3/envs/kaggle-313/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


logit: OOF AUC=0.8702 ACC@0.5=0.8238
rf: OOF AUC=0.8751 ACC@0.5=0.8260
rf: OOF AUC=0.8751 ACC@0.5=0.8260
hgb: OOF AUC=0.8608 ACC@0.5=0.8215
hgb: OOF AUC=0.8608 ACC@0.5=0.8215


In [ ]:
# 7) Weight + threshold search (optimize Accuracy)
best = {"acc": -1.0, "w": (0.0,0.0,1.0), "t": 0.5, "auc": -1.0}
grid = np.linspace(0.0, 1.0, 21)
p_l, p_r, p_h = p_oof['logit'], p_oof['rf'], p_oof['hgb']
for wl in grid:
    for wr in grid:
        if wl + wr > 1.0:
            continue
        wh = 1.0 - wl - wr
        blend = wl*p_l + wr*p_r + wh*p_h
        for t in np.linspace(0.3, 0.7, 41):
            preds = (blend >= t).astype(int)
            acc = metrics.accuracy_score(y, preds)
            if acc > best['acc']:
                best['acc'] = float(acc)  # Ensure 'acc' is stored as a float
                best['w'] = (float(wl), float(wr), float(wh))  # Corrected to use float values directly
                best['t'] = float(t)
                best['auc'] = float(metrics.roc_auc_score(y, blend))
print("Best weights (logit, rf, hgb)=", 
      best['w'], " threshold=", 
      round(best['t'], 4), " ACC=", 
      round(best['acc'], 4), " AUC=", 
      round(best['auc'], 4))
best_weights = best['w']; best_threshold = best['t']

Best weights (logit, rf, hgb)= (0.8, 0.2, -5.551115123125783e-17)  threshold= 0.6199999999999999  ACC= 0.8384  AUC= 0.8752


In [8]:
# 8) Fit base models on full training
for name, mdl in models.items():
    mdl.fit(X, y)
print("Fitted base models on full data.")

Fitted base models on full data.


In [9]:
# 9) Predict test and save submission
probs_test = {}
for name, mdl in models.items():
    probs_test[name] = mdl.predict_proba(X_test)[:, 1]
wl, wr, wh = best_weights
blend_test = wl*probs_test['logit'] + wr*probs_test['rf'] + wh*probs_test['hgb']
labels_test = (blend_test >= best_threshold).astype(int)
submission = pd.DataFrame({ 'PassengerId': test_df_fe['PassengerId'], 'Survived': labels_test })
submission.to_csv(SUBMISSION_PATH, index=False)
print("Saved submission to:", SUBMISSION_PATH)
submission.head(10)

Saved submission to: /home/atul-kumar/workspace/kaggle/titanic/data/submission-ensemble.csv


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
5,897,0
6,898,0
7,899,0
8,900,1
9,901,0


In [10]:
# 10) Save ensemble artifact
artifact_path = os.path.join(MODEL_DIR, 'titanic-ensemble.joblib')
metadata = {
    'weights': {'logit': best_weights[0], 'rf': best_weights[1], 'hgb': best_weights[2]},
    'threshold': best_threshold
}
joblib.dump({'models': models, 'metadata': metadata}, artifact_path)
print("Saved ensemble artifact to:", artifact_path)
metadata

Saved ensemble artifact to: /home/atul-kumar/workspace/kaggle/titanic/data/models/titanic-ensemble.joblib


{'weights': {'logit': 0.8, 'rf': 0.2, 'hgb': -5.551115123125783e-17},
 'threshold': 0.6199999999999999}

In [11]:
# 11) Reproducibility
os.environ['PYTHONHASHSEED'] = str(RANDOM_STATE)
os.environ.setdefault('OMP_NUM_THREADS', '1')
print("Seeds and environment set.")

Seeds and environment set.
